# Rung 15 — the structured counting target

**One variable against rung 06: the assistant `content` of `answer_format == "number"` rows.**
`system`, `user`, `images`, pixels, recipe, seed, epochs and every swift flag are unchanged —
`diff_vs_rung06` proves the argv is identical, and a sha256 proves the flag-OFF export is
byte-identical to rung 06's.

| Notebook | Rung | bucket_mean (canonical) | Verdict |
|---|---|---|---|
| `../00-baseline/00_zeroshot_qwen3vl.ipynb` | 00 | 0.2557 | baseline |
| `../02-lora-sft/02_lora_sft.ipynb` | 02 | 0.5486 | PASS — LoRA on the LLM |
| `../06-vit-lora/06_vit_lora.ipynb` | 06 | **0.5667** | 🟡 PARTIAL — **the control for this rung** |
| `15_count_target.ipynb` | 15 | *(this run)* | — |

**The question.** Rung 02/06 train `number` rows on a bare integer (`"3"`): the whole gradient
lands on one token, and the model learns a prior over that token rather than a count. Measured
consequence — bias −0.66, golds 3–4 share modal prediction 2, golds 5–8 share modal prediction 4,
`number` margin OOD decaying +0.015 → +0.013 → **+0.000** across epochs (rung 06). Is that the
TASK, or the TARGET FORMAT?

**The target** (v05 Gautam 2025 field names verbatim; label-first per v28 Guo 2025 — see
`_models/count_target.py` and `context/15-count-target/CONTEXT.md` §Why this format):

```
{"label": "Clips", "counts": 2}
```

🔴 **This notebook reports numbers; it does not decide anything.** The decision rule, the arms and
the stopping tiers are pre-registered in `context/15-count-target/CONTEXT.md`, written before any
number existed. A threshold edited after seeing a number stops being a gate and becomes a story.

🔴 **Do not judge this run with `acc_number`** — it averages 8 val templates whose floors run
0.24–1.00, four of them degenerate (data card rung 08 §3; `context/RULES.md` §12). Read the
`number` **margin** over the template-aware floor, per distribution, **and the per-template table**.

In [ ]:
# papermill parameters
SMOKE = True           # True -> tiny export + 2 train steps + 40-question eval: proves the chain
SMOKE_STEPS = 2
COUNT_TARGET = True    # 🎯 THE VARIABLE. False -> byte-identical to rung 06 (gated by sha256)
RUN_TAG = "eval_best"  # eval sub-dir under the run dir, same name rung 06 used

In [ ]:
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

# `swift` is shelled out to by merge_checkpoint; a Jupyter/papermill kernel does not inherit
# the env's bin/ on PATH, and it must be THIS interpreter's bin so CLI and kernel match.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP / "_models", EXP / "_tools", REPO / "src",
          REPO / "experiments/06-vit-lora/_models", REPO / "experiments/02-lora-sft/_models"):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

import count_target as ct
from count_target import CountTargetConfig, main, diff_vs_rung06, list_checkpoints, merge_checkpoint

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly — an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path("/workspace/orena-data"), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no orena-data found (pod: /workspace/orena-data)"

RUNG06_RUN = REPO / "experiments/06-vit-lora/runs/06_vit_lora_v1"
RUNG06_BUCKET_MEAN = 0.566707285167315   # committed, RESULTS.csv — never re-run, never re-derived

cfg = CountTargetConfig(
    data_root=DATA_ROOT,
    exp_dir=REPO / "experiments" / "15-count-target",
    run_name=("15_count_target_smoke" if SMOKE else "15_count_target_v1"),
    manifest_path=REPO / "experiments/splits/frame_ood_v1.csv",
    count_target=COUNT_TARGET,          # 🎯 THE VARIABLE
    rung06_run_dir=RUNG06_RUN,
    parse_answers=COUNT_TARGET,         # the parser is wired iff the target changed
    smoke=SMOKE,
    smoke_max_steps=SMOKE_STEPS,
)
print("run_dir     :", cfg.run_dir)
print("count_target:", cfg.count_target, " parse_answers:", cfg.parse_answers)
print("data_root   :", cfg.data_root)

## Gates that cost nothing — run them all before the GPU is touched

`context/RULES.md` §7: **gates RAISE and are never disabled.** A gate that fires is a finding.

In [ ]:
# G-ARGV — the training recipe must be rung 06's, flag for flag. Not a promise: rung 06's own
# `_swift_args` is called on both configs and the argv is diffed. Must be EMPTY.
diff = diff_vs_rung06(cfg)
assert diff == {}, (
    f"G-ARGV FAILED: rung 15 differs from rung 06 in {sorted(diff)} -> {diff}. The ONLY "
    "pre-registered difference is the bytes of train.jsonl; a flag difference means this run "
    "answers a different question."
)
print("OK G-ARGV: swift argv identical to rung 06 (0 flags differ)")

In [ ]:
# 🔴 G-PARSE — the parser is the gate this rung stands on, and it is fully checkable offline.
# Exhaustive: every REAL `number` template (train ∪ val, read from the data card's committed
# tables) x every gold 0..12, plus an enumerated malformed battery. Also proves SAFE_ANSWER is
# not the modal answer of any template, so a malformed generation COSTS us instead of quietly
# inheriting the template floor.
import test_count_target as t15
t15.main()

## Export — and the two byte-identity gates

🔴 **Flag OFF must produce rung 06's `train.jsonl` byte for byte** (wave spec non-negotiable #4).
Proven by sha256 against rung 06's own committed export, never asserted in prose.

🔴 **Flag ON must touch only `number` rows.** `assert_nonnumber_lines_identical` copies every
non-`number` line verbatim (never re-serialised) and then re-reads both files to prove it.

Both gates fire *inside* `main(cfg, stage="export")` — a broken rewrite cannot reach the GPU.

In [ ]:
# Export. Rung 02's OWN exporter runs first (identical frames, identical ordering, identical
# bytes); the `number`-row rewrite is applied on top of the file it produced.
# Frames come from the shared /workspace/frames_cache — a frame exists ONCE and is called from
# there; rung 05 duplicated 2.6 GB by ignoring this.
t0 = time.perf_counter()
main(cfg, stage="export")
n_rows = sum(1 for _ in open(cfg.train_jsonl, encoding="utf-8"))
print(f"train.jsonl -> {cfg.train_jsonl}  ({n_rows} examples, {time.perf_counter()-t0:.0f}s)")
print("sha256      :", ct.sha256_of(cfg.train_jsonl))
if cfg.count_target:
    tg = [json.loads(l) for l in open(cfg.count_targets_jsonl, encoding="utf-8")]
    print(f"rewritten   : {len(tg)} `number` rows ({100*len(tg)/n_rows:.1f}% of the export)")
    print("examples    :")
    for r in tg[:3]:
        print("   ", r["target"], "   <-", r["question"][:70])

In [ ]:
# G-REGEX — `is_count_question` must agree with `answer_format` on EVERY item of BOTH splits.
# Two fatal directions: a `number` row we do not rewrite (the variable silently shrinks) and a
# non-`number` row we do (a second variable appears). templates_val.csv carries an OPEN-ENDED
# "How many radiopaque clips are visible in this video?" — a loose regex would swallow it.
from frame.config import BaselineConfig
from frame.data import load_frame_items
_items = load_frame_items(BaselineConfig(data_root=DATA_ROOT), splits=("train", "test"))
print("G-REGEX:", ct.assert_count_regex_matches_formats(_items))

In [ ]:
# 🔴 G-OFF — the flag-OFF export must be byte-identical to rung 06's. Costs one extra export
# (frames are already cached, so it is I/O, not GPU). SKIPPED IN SMOKE only because the smoke
# subsample makes the comparand a different set of rows; it is MANDATORY before the full run.
if SMOKE:
    print("SMOKE: G-OFF deferred — it must pass on the full export before training is launched.")
else:
    scratch = Path("/workspace/tmp/g_off_15")   # /workspace root stays clean; deleted below
    print("G-OFF:", ct.assert_flag_off_identical(cfg, scratch))
    import shutil; shutil.rmtree(scratch, ignore_errors=True)
    print("scratch removed (CONSTITUTION §IX.1)")

## Train — rung 06's recipe, unchanged

`swift sft` is rung 06's `_train`, called on this config. Same LoRA (r=8, α=32, all-linear,
`freeze_vit=False`, `freeze_aligner=True`), same LR 2e-5, same cosine schedule, same seed 42,
same `MAX_PIXELS`, same `save_strategy=epoch`.

**STOP RULES.** OOM → STOP; do NOT lower `max_pixels`/batch/LR — each is a second variable.
Loss diverges → do NOT touch the LR; report it. The broken-run guard aborts only on NaN/inf or
an epoch-1 eval that bought nothing, and it is a pure stdout observer.

In [ ]:
t0 = time.perf_counter()
main(cfg, stage="train")
print(f"\ntrained in {(time.perf_counter()-t0)/60:.1f} min")
print("checkpoints:", [c.name for c in list_checkpoints(cfg)])
print("G1 (swift's own log):", json.dumps(ct.read_g1(cfg), indent=1))

## Per-epoch merge + eval — **every** epoch, never last-by-default

`context/RULES.md` §6 and the wave spec: *training ERASES counting, and `acc_OOD` checkpoint
selection picks the checkpoint that erased more.* So every epoch is merged, evaluated and
reported; the selection happens in the write-up against the pre-registered rule, not here.

The parser is wired into the answer path by `BaselineConfig.answer_postprocess`
(`src/frame/config.py:46`), which `frame.engine.QwenFrameEngine.predict` applies at
`src/frame/engine.py:129` — after the generation, before `Response.content` exists and before the
SDK's format verification. It has to happen there: `Number.verify` requires `isdigit()` and
`Evaluator._evaluate_single` marks anything else **incorrect** — a structured answer reaching the
judge is a guaranteed 0. `answer_postprocess=None` (the default, and what the control arm passes)
skips the call entirely, so the OFF path is byte-identical. Non-count questions take the identical
path they always have: `parse_count(..., is_count=False)` returns the generation untouched.

⚠️ Disk: each merged 8B bf16 checkpoint is ~17 GB. Each is deleted right after its eval.

In [ ]:
from frame.config import BaselineConfig
from frame.run import run_baseline
from frame import ledger, metrics

gold = ledger.gold_from_frame_parquets(DATA_ROOT)
epoch_rows, strats, parse_stats = [], {}, {}

for ck in list_checkpoints(cfg):
    tag = f"{RUN_TAG}__{ck.name}"
    print(f"\n{'='*70}\n{ck.name}\n{'='*70}")
    merged = merge_checkpoint(cfg, ck)
    # `postprocess` is the shared hook (src/frame/config.py:46 -> engine.py:129), not a
    # monkeypatch of the engine. Flag OFF yields None, which is the field's default, so the
    # control arm's answer path is byte-identical.
    with ct.count_answer_hook(enabled=cfg.parse_answers,
                              log_path=cfg.run_dir / tag / "parse_log.jsonl") as (postprocess, plog):
        bcfg = BaselineConfig(
            data_root=DATA_ROOT, model_path=merged, out_dir=cfg.run_dir, run_name=tag,
            max_pixels=cfg.max_pixels, seed=cfg.seed, n_eval=40 if SMOKE else None,
            answer_postprocess=postprocess,
        )
        assert (bcfg.answer_postprocess is not None) == cfg.parse_answers, (
            "the parser is not wired as configured — an unwired post-processor "
            "is indistinguishable from a model that cannot count")
        t0 = time.perf_counter()
        run_baseline(bcfg)
        print(f"eval done in {(time.perf_counter()-t0)/60:.1f} min")
        parse_stats[ck.name] = ct.malformed_stats(plog)
    print("malformed:", json.dumps(parse_stats[ck.name], indent=1))

    res = pd.read_csv(cfg.run_dir / tag / "results.csv")
    missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
    assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; margins would be inflated"
    metrics.assert_no_dup_qid(res); metrics.assert_ood_from_qid(res); metrics.assert_all_rows_grouped(res)
    s = metrics.stratified_report(res, gold=gold)
    metrics.assert_floors_vs_eval_set(s)
    ct.assert_safe_answer_not_modal(gold, res)   # the fallback must still cost us
    strats[ck.name] = s
    if not SMOKE:
        ledger.register_run(cfg.run_dir / tag, s, experiment="15-count-target",
                            run=f"{cfg.run_name}__{ck.name}",
                            model=f"Qwen3-VL-8B + LoRA r8 ViT+LLM, count target ({ck.name})",
                            extra={"count_target": cfg.count_target,
                                   "malformed": parse_stats[ck.name]})
    bf = pd.DataFrame(s["by_format"]); num = bf[bf.answer_format == "number"].set_index("distribution")
    epoch_rows.append({
        "checkpoint": ck.name, "bucket_mean": s["bucket_mean"],
        "acc_ID": s["acc_ID"], "acc_OOD": s["acc_OOD"],
        "margin_ID": s["margin_ID"], "margin_OOD": s["margin_OOD"],
        "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
        "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
        "malformed_rate": parse_stats[ck.name]["malformed_rate"],
    })
    # 17 GB per merged checkpoint — rung 12 hit exactly this and had to add the discipline mid-run.
    import shutil; shutil.rmtree(merged, ignore_errors=True)
    print("merged weights deleted:", merged)

epochs = pd.DataFrame(epoch_rows)
print("\n", epochs.to_string(index=False))

## The pre-registered read

**Primary** = `bucket_mean` (canonical, `frame.metrics`). **The target of this rung** = the
`number` margin over the template-aware floor, in BOTH distributions, at EVERY epoch.

**WIN** (pre-registered, `context/15-count-target/CONTEXT.md`): `number` margin up vs rung 06 in
**both** ID and OOD, with a paired video-clustered CI on the delta excluding 0, **and** no other
format's margin down by more than 0.02.

The paired delta on `number` correctness **is** the margin delta: the floor depends only on the
gold answers and the templates, which are identical across arms, so it cancels exactly in the
difference. That is why `metrics.paired_delta_ci` (video-clustered — effective n ≈ 38 videos, not
6252 questions) is the right instrument and no new estimator is needed.

In [ ]:
# Paired, video-clustered delta vs rung 06 — same questions, same frames, same judge protocol.
R06 = RUNG06_RUN / "eval_best" / "results.csv"
assert R06.exists(), f"rung 06's scored predictions not found at {R06} — the control is required"
r06 = pd.read_csv(R06)[["qID", "video", "answer_format", "correctness"]].rename(
    columns={"correctness": "correct_a"})

# Checkpoint selection = rung 06's rule, unchanged: max acc_OOD (context/RULES.md §6).
# Changing the criterion would be a SECOND variable — rung 06b says so in as many words. Every
# epoch is reported above regardless, so a reader can see what any other rule would have picked.
BEST = epochs.sort_values("acc_OOD", ascending=False).iloc[0]["checkpoint"]
res = pd.read_csv(cfg.run_dir / f"{RUN_TAG}__{BEST}" / "results.csv")[["qID", "correctness"]].rename(
    columns={"correctness": "correct_b"})
pair = r06.merge(res, on="qID", how="inner")
assert len(pair) == len(r06), f"paired join lost {len(r06)-len(pair)} questions — arms must share the eval set"
pair["distribution"] = pair["qID"].map(metrics._dist_from_qid)

paired = []
for (fmt, dist), sub in pair.groupby(["answer_format", "distribution"]):
    d = metrics.paired_delta_ci(sub, n_boot=4000, seed=42)
    paired.append({"answer_format": fmt, "distribution": dist, **d,
                   "excludes_0": bool(d["ci_low"] > 0 or d["ci_high"] < 0)})
paired = pd.DataFrame(paired).sort_values(["answer_format", "distribution"])
print(paired.to_string(index=False))

In [ ]:
# 🔴 The per-template `number` table — `acc_number` pooled is NOT interpretable (RULES §12;
# data card rung 08 §3: 8 val templates, floors 0.24–1.00, FOUR of them degenerate). A gain
# confined to the two big templates can be buried in the pool, and a wobble in a degenerate
# template can masquerade as progress. This is the table the verdict is read from.
res_best = pd.read_csv(cfg.run_dir / f"{RUN_TAG}__{BEST}" / "results.csv")
tpl = ct.number_margin_by_template(res_best, gold)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
print(tpl.to_string(index=False))
if not SMOKE:
    tpl.to_csv(EXP / "RESULTS_number_by_template.csv", index=False)

In [ ]:
# Ledger-shaped RESULTS.csv (one row per run — `frame.ledger` reads an `arm` column as a run
# name, src/frame/ledger.py:67, so per-epoch/per-arm detail goes to SEPARATE files or the
# shared ledger is poisoned).
if not SMOKE:
    s = strats[BEST]
    bf = pd.DataFrame(s["by_format"])
    def _acc(f):
        sub = bf[bf.answer_format == f]
        return float((sub.accuracy * sub.n).sum() / sub.n.sum()) if len(sub) else float("nan")
    row = {
        "run": cfg.run_name,
        "model": f"Qwen3-VL-8B-Instruct + LoRA r8 ViT+LLM, structured count target ({BEST})",
        "bucket_mean": s["bucket_mean"], "acc_ID": s["acc_ID"], "acc_OOD": s["acc_OOD"],
        "floor_ID": s["floor_ID"], "floor_OOD": s["floor_OOD"],
        "margin_ID": s["margin_ID"], "margin_OOD": s["margin_OOD"],
        "acc_fo_class": _acc("fo_class"), "acc_number": _acc("number"),
        "acc_binary": _acc("binary"), "acc_open_ended": _acc("open_ended"),
        "acc_multiple_choice": _acc("multiple_choice"),
        "n_questions": int(len(res_best)), "date": pd.Timestamp.utcnow().strftime("%Y-%m-%d"),
        "notes": (f"single variable vs 06-vit-lora: assistant content of answer_format=='number' "
                  f"rows only ({{'label':…,'counts':N}}, v05 field names). Parser malformed rate "
                  f"{parse_stats[BEST]['malformed_rate']:.4f} (Perek 2026 vanilla CoT-SFT ref 0.0476). "
                  f"Per-epoch table in RESULTS_epochs.csv; per-template `number` margin in "
                  f"RESULTS_number_by_template.csv; paired CIs in runs/{cfg.run_name}/paired_delta.csv. "
                  f"🔴 acc_number is NOT interpretable pooled — read the per-template margins."),
    }
    pd.DataFrame([row]).to_csv(EXP / "RESULTS.csv", index=False)
    epochs.to_csv(EXP / "RESULTS_epochs.csv", index=False)
    paired.to_csv(cfg.run_dir / "paired_delta.csv", index=False)
    (cfg.run_dir / "parse_stats.json").write_text(json.dumps(parse_stats, indent=2), encoding="utf-8")
    print("wrote RESULTS.csv, RESULTS_epochs.csv, RESULTS_number_by_template.csv, paired_delta.csv")

print(f"\nrung 06 bucket_mean {RUNG06_BUCKET_MEAN:.4f}  ->  rung 15 "
      f"{strats[BEST]['bucket_mean']:.4f}   (delta {strats[BEST]['bucket_mean']-RUNG06_BUCKET_MEAN:+.4f})")
print("\n🔴 The verdict is read by a human against the pre-registered rule in "
      "context/15-count-target/CONTEXT.md. This notebook does not decide.")